# 第六课：数据加载与预处理

## 为什么需要专门的数据加载工具？

实际深度学习项目中，数据往往：
- 数据量大（GB 级别），无法一次性加载到内存
- 需要预处理（归一化、裁剪、翻转等）
- 需要随机打乱、分批次训练

PyTorch 提供了 `Dataset` + `DataLoader` 的优雅方案。

## 核心组件

| 组件 | 职责 |
|------|------|
| `Dataset` | 定义如何获取**一条**数据（读文件、预处理等） |
| `DataLoader` | 负责批量加载、打乱、并行读取 |
| `Transform` | 定义数据预处理流水线 |

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## 1. TensorDataset：最简单的数据集

当数据已经是 Tensor 时，用 `TensorDataset` 最方便。

In [ ]:
# 创建 TensorDataset
X = torch.randn(1000, 5)
y = (X[:, 0] + X[:, 1] > 0).long()

dataset = TensorDataset(X, y)

print(f"数据集大小: {len(dataset)}")
sample_x, sample_y = dataset[0]
print(f"第一条数据: x={sample_x}, y={sample_y}")

batch_x, batch_y = dataset[0:4]
print(f"\n前4条数据:")
print(f"  X shape: {batch_x.shape}")
print(f"  y: {batch_y}")

## 2. DataLoader：批量加载器

DataLoader 是数据加载的核心，负责：
- 分批（batching）
- 打乱（shuffling）
- 并行加载（num_workers）
- 自动补齐最后一批（drop_last）

In [ ]:
dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    drop_last=False
)

print(f"总样本数: {len(dataset)}")
print(f"批次数: {len(dataloader)}")

for batch_idx, (X_batch, y_batch) in enumerate(dataloader):
    if batch_idx < 3 or batch_idx == len(dataloader) - 1:
        print(f"Batch {batch_idx}: X shape={X_batch.shape}, y shape={y_batch.shape}")
    elif batch_idx == 3:
        print("...")

In [ ]:
print("=== DataLoader 关键参数 ===")
print()
print("batch_size: 每批样本数")
print("  - 太小（如1）：训练不稳定，速度慢")
print("  - 太大（如全部）：内存不够，泛化可能变差")
print("  - 常用值：32, 64, 128, 256")
print()
print("shuffle: 是否打乱")
print("  - 训练集：True（重要！否则模型会记住顺序）")
print("  - 测试集：False（顺序不影响评估）")
print()
print("num_workers: 数据加载的子进程数")
print("  - 0：主进程加载（简单但慢）")
print("  - >0：多进程并行加载（快，但Windows下可能有坑）")
print("  - 建议：Linux/Mac 用 2-8，Windows 用 0")
print()
print("drop_last: 是否丢弃最后不完整的批次")
print("  - False（默认）：保留，最后一批可能较小")
print("  - True：丢弃，所有批次大小一致")

## 3. 自定义 Dataset

当数据不是简单的 Tensor（如图片文件、CSV 文件），需要自定义 Dataset。

必须实现三个方法：
1. `__init__`: 初始化，加载数据元信息
2. `__len__`: 返回数据集大小
3. `__getitem__`: 根据索引返回一条数据

In [ ]:
class NumpyDataset(Dataset):
    def __init__(self, X_np, y_np, transform=None):
        self.X = torch.from_numpy(X_np).float()
        self.y = torch.from_numpy(y_np).long()
        self.transform = transform
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

X_np = np.random.randn(500, 3).astype(np.float32)
y_np = np.random.randint(0, 3, 500).astype(np.int64)

np_dataset = NumpyDataset(X_np, y_np)
print(f"NumpyDataset 大小: {len(np_dataset)}")
print(f"第一条: x={np_dataset[0][0]}, y={np_dataset[0][1]}")

In [ ]:
import os
from PIL import Image

class ImageDataset(Dataset):
    """
    自定义图片数据集的典型结构
    目录结构:
        root/
            class_0/
                img001.jpg
            class_1/
                img003.jpg
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((
                        os.path.join(cls_dir, fname),
                        self.class_to_idx[cls]
                    ))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print("ImageDataset 定义完成")
print("用法: dataset = ImageDataset('path/to/data', transform=transforms)")
print("关键: __getitem__ 中做懒加载，只在实际需要时读取文件")

In [ ]:
class AugmentedDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = X
        self.y = y
        self.augment = augment
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        y = self.y[idx]
        if self.augment:
            if torch.rand(1) > 0.5:
                x += 0.1 * torch.randn_like(x)
            if torch.rand(1) > 0.5:
                mask = torch.rand_like(x) > 0.2
                x = x * mask.float()
        return x, y

X_demo = torch.randn(100, 5)
y_demo = torch.randint(0, 2, (100,))

aug_dataset = AugmentedDataset(X_demo, y_demo, augment=True)
x1, y1 = aug_dataset[0]
x2, y2 = aug_dataset[0]
print(f"同一条数据两次获取:")
print(f"  第1次: {x1}")
print(f"  第2次: {x2}")
print(f"  是否相同: {torch.equal(x1, x2)}")

## 4. 数据集划分：训练集 / 验证集 / 测试集

In [ ]:
full_dataset = TensorDataset(torch.randn(1000, 10), torch.randint(0, 5, (1000,)))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"原始数据集: {len(full_dataset)}")
print(f"训练集: {len(train_dataset)} ({len(train_dataset)/len(full_dataset)*100:.0f}%)")
print(f"验证集: {len(val_dataset)} ({len(val_dataset)/len(full_dataset)*100:.0f}%)")
print(f"测试集: {len(test_dataset)} ({len(test_dataset)/len(full_dataset)*100:.0f}%)")

In [ ]:
torch.manual_seed(42)
X = torch.randn(1000, 10)
y = torch.randint(0, 5, (1000,))

indices = torch.randperm(len(X))

train_idx = indices[:700]
val_idx = indices[700:850]
test_idx = indices[850:]

train_dataset = TensorDataset(X[train_idx], y[train_idx])
val_dataset = TensorDataset(X[val_idx], y[val_idx])
test_dataset = TensorDataset(X[test_idx], y[test_idx])

print(f"训练集: {len(train_dataset)}")
print(f"验证集: {len(val_dataset)}")
print(f"测试集: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"训练批次数: {len(train_loader)}")
print(f"验证批次数: {len(val_loader)}")
print(f"测试批次数: {len(test_loader)}")
print()
print("训练集 shuffle=True（每个epoch顺序不同）")
print("验证/测试集 shuffle=False（顺序不影响评估）")
print("验证/测试集 batch_size 可以更大（不需要反向传播）")

## 5. Transforms：数据预处理流水线

PyTorch 的 `torchvision.transforms` 提供了丰富的图像预处理功能。

In [ ]:
try:
    from torchvision import transforms
    has_torchvision = True
except ImportError:
    has_torchvision = False
    print("未安装 torchvision，跳过图像 Transforms 部分")

if has_torchvision:
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    print("训练集 Transform:")
    print(train_transform)
    print("\n测试集 Transform:")
    print(test_transform)

In [ ]:
if has_torchvision:
    from PIL import Image
    
    test_img = Image.fromarray(np.random.randint(0, 256, (300, 300, 3), dtype=np.uint8))
    
    augment_vis = transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(30),
    ])
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes[0, 0].imshow(test_img)
    axes[0, 0].set_title('Original')
    axes[0, 0].axis('off')
    
    for i, ax in enumerate(axes.flatten()[1:]):
        augmented = augment_vis(test_img)
        ax.imshow(augmented)
        ax.set_title(f'Augmented {i+1}')
        ax.axis('off')
    
    plt.suptitle('Data Augmentation Examples', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("每次获取同一条数据，增强结果都不同！")
    print("这等价于扩充了训练数据量，有助于防止过拟合")

## 6. 自定义 Transform

In [ ]:
class AddGaussianNoise:
    def __init__(self, mean=0.0, std=0.1):
        self.mean = mean
        self.std = std
    
    def __call__(self, tensor):
        return tensor + torch.randn_like(tensor) * self.std + self.mean

class ClipRange:
    def __init__(self, min_val=0.0, max_val=1.0):
        self.min_val = min_val
        self.max_val = max_val
    
    def __call__(self, tensor):
        return tensor.clamp(self.min_val, self.max_val)

class Standardize:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean)
        self.std = torch.tensor(std)
    
    def __call__(self, tensor):
        return (tensor - self.mean) / self.std

if has_torchvision:
    custom_transform = transforms.Compose([
        AddGaussianNoise(std=0.05),
        ClipRange(0, 1),
    ])
    
    x = torch.ones(3, 5) * 0.5
    x_transformed = custom_transform(x)
    print(f"原始: {x[0, :3]}")
    print(f"变换后: {x_transformed[0, :3]}")
else:
    print("需要 torchvision 才能运行此示例")

## 7. collate_fn：自定义批次组装

当数据集中的样本长度不一致（如文本序列），需要自定义 `collate_fn`。

In [ ]:
variable_lengths = [
    (torch.randn(3,), 0),
    (torch.randn(5,), 1),
    (torch.randn(2,), 0),
    (torch.randn(4,), 1),
]

try:
    default_loader = DataLoader(variable_lengths, batch_size=4)
    for batch in default_loader:
        pass
except RuntimeError as e:
    print(f"默认 DataLoader 报错: {e}")
    print("原因: 变长张量无法直接 stack")

In [ ]:
def pad_collate_fn(batch):
    sequences, labels = zip(*batch)
    max_len = max(len(seq) for seq in sequences)
    padded_seqs = torch.zeros(len(sequences), max_len)
    lengths = []
    for i, seq in enumerate(sequences):
        padded_seqs[i, :len(seq)] = seq
        lengths.append(len(seq))
    labels = torch.tensor(labels)
    lengths = torch.tensor(lengths)
    return padded_seqs, labels, lengths

padded_loader = DataLoader(variable_lengths, batch_size=4, collate_fn=pad_collate_fn)
for padded_batch, labels, lengths in padded_loader:
    print(f"填充后的序列:\n{padded_batch}")
    print(f"标签: {labels}")
    print(f"原始长度: {lengths}")

In [ ]:
def list_collate_fn(batch):
    sequences, labels = zip(*batch)
    return list(sequences), torch.tensor(labels)

list_loader = DataLoader(variable_lengths, batch_size=2, collate_fn=list_collate_fn)
for seqs, labels in list_loader:
    print(f"序列列表: {[s.shape for s in seqs]}")
    print(f"标签: {labels}")

## 8. 完整训练流程：带验证集

In [ ]:
torch.manual_seed(42)
n_samples = 2000
n_features = 10
n_classes = 5

X = torch.randn(n_samples, n_features)
W_true = torch.randn(n_features, n_classes)
logits_true = X @ W_true
y = logits_true.argmax(dim=1)

full_dataset = TensorDataset(X, y)
train_size = int(0.7 * n_samples)
val_size = int(0.15 * n_samples)
test_size = n_samples - train_size - val_size

train_set, val_set, test_set = random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=128, shuffle=False)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

print(f"训练集: {len(train_set)}, 验证集: {len(val_set)}, 测试集: {len(test_set)}")

In [ ]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 5)
        )
    
    def forward(self, x):
        return self.net(x)

model = Classifier()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
train_losses = []
val_losses = []
train_accs = []
val_accs = []
best_val_acc = 0.0

for epoch in range(30):
    model.train()
    epoch_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in train_loader:
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)
        correct += (logits.argmax(dim=1) == y_batch).sum().item()
        total += X_batch.size(0)
    
    train_loss = epoch_loss / total
    train_acc = correct / total
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            val_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            val_total += X_batch.size(0)
    
    val_loss_avg = val_loss / val_total
    val_acc = val_correct / val_total
    val_losses.append(val_loss_avg)
    val_accs.append(val_acc)
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}: "
              f"train_loss={train_loss:.4f}, train_acc={train_acc:.4f} | "
              f"val_loss={val_loss_avg:.4f}, val_acc={val_acc:.4f}")

print(f"\n最佳验证准确率: {best_val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, label='Train Loss')
axes[0].plot(val_losses, label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_accs, label='Train Acc')
axes[1].plot(val_accs, label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print("如果 train_loss 持续下降但 val_loss 开始上升 -> 过拟合")
print("如果两者都很高 -> 欠拟合")

In [ ]:
model.load_state_dict(best_model_state)
model.eval()

test_correct = 0
test_total = 0
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        test_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        test_total += X_batch.size(0)

test_acc = test_correct / test_total
print(f"测试集准确率: {test_acc:.4f}")
print(f"验证集最佳准确率: {best_val_acc:.4f}")
print()
print("报告测试集准确率，而不是验证集准确率！")
print("验证集用于选择超参数和模型，测试集只用于最终评估")

## 9. 数据归一化/标准化

In [ ]:
torch.manual_seed(42)
n = 1000
X_raw = torch.cat([
    torch.randn(n, 1) * 100,
    torch.randn(n, 1) * 0.01,
    torch.randn(n, 1) * 10,
], dim=1)

W_true = torch.tensor([1.0, 5.0, -2.0])
y_raw = (X_raw @ W_true + 0.5 * torch.randn(n) > 0).long()

print("原始数据统计:")
for i in range(3):
    print(f"  特征{i}: mean={X_raw[:, i].mean():.2f}, std={X_raw[:, i].std():.2f}")

X_mean = X_raw.mean(dim=0)
X_std = X_raw.std(dim=0)
X_norm = (X_raw - X_mean) / X_std

print("\n标准化后:")
for i in range(3):
    print(f"  特征{i}: mean={X_norm[:, i].mean():.4f}, std={X_norm[:, i].std():.4f}")

In [ ]:
def train_and_compare(X, y, epochs=100):
    dataset = TensorDataset(X, y)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)
    model = nn.Sequential(nn.Linear(3, 16), nn.ReLU(), nn.Linear(16, 2))
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        for X_batch, y_batch in loader:
            loss = criterion(model(X_batch), y_batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
    return losses

raw_losses = train_and_compare(X_raw, y_raw)
norm_losses = train_and_compare(X_norm, y_raw)

plt.figure(figsize=(10, 5))
plt.plot(raw_losses, label='Raw Data', alpha=0.7)
plt.plot(norm_losses, label='Normalized Data', alpha=0.7)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Effect of Normalization on Training')
plt.legend()
plt.grid(True)
plt.show()

print(f"原始数据最终 loss: {raw_losses[-1]:.4f}")
print(f"标准化数据最终 loss: {norm_losses[-1]:.4f}")
print("标准化让所有特征在同一尺度，梯度下降更高效！")

---
## 总结

### Dataset 三种创建方式

| 方式 | 适用场景 |
|------|----------|
| `TensorDataset` | 数据已经是 Tensor |
| 自定义 `Dataset` | 从文件加载、需要预处理 |
| `torchvision.datasets` | 常见公开数据集 |

### DataLoader 关键参数

```python
DataLoader(dataset, batch_size=64, shuffle=True, num_workers=0, drop_last=False)
```

### 完整训练流程模板

```python
for epoch in range(epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        loss = criterion(model(X_batch), y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    model.eval()
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            pass
    
    if val_acc > best_val_acc:
        best_model_state = model.state_dict()
```

### 关键原则
- 训练集打乱，验证/测试集不打乱
- 数据增强只用于训练集
- 验证集选模型，测试集做最终评估
- 数据标准化加速收敛
- model.train() / model.eval() 别忘了切换